# Question 4
Build a full SCD Type 2 table: implement the MERGE that closes out old records (setting end_date and is_current) and inserts new versions when a tracked column changes. 

In [0]:
from pyspark.sql.functions import *

# 1. Source Path (Jahan tumhari Raw CSV files aayengi)
raw_source_path = "/Volumes/cyntexa_dev/day_8/my_volume/customers/"

# 2. Schema Checkpointing Path (Auto Loader state track karne ke liye)
checkpoint_path = "/Volumes/cyntexa_dev/day_8/my_volume/bronze_customers/"
schema_path = "/Volumes/cyntexa_dev/day_8/my_volume/schema_location/"
# 3. Auto Loader Read Stream (Zero Cleaning - Pure Raw Data Capture)
raw_stream_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation",schema_path) \
    .option("header", "true") \
    .load(raw_source_path) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file_name", col("_metadata.file_name")) \
    .withColumn("source_file_path", col("_metadata.file_path"))

# 4. Write Stream to Bronze Table (Trigger Once taaki ek batch me run ho kar rukk jaaye)
query = raw_stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .toTable("cyntexa_dev.day_8.bronze_customers")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cyntexa_dev.day_8.silver_customers_scd2 (
    customer_id STRING,
    first_name STRING,
    last_name STRING,
    email STRING,
    city STRING,
    signup_date STRING,
    start_date TIMESTAMP,
    end_date TIMESTAMP,
    is_current BOOLEAN
) USING DELTA;

In [0]:
from pyspark.sql.functions import col, trim, lower, row_number, to_timestamp
from pyspark.sql.window import Window

# Bronze read
bronze_df = spark.read.table("cyntexa_dev.day_8.bronze_customers")

# Basic Cleaning (Null Check + String Trimming)
cleaned_df = bronze_df \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("first_name", trim(col("first_name"))) \
    .withColumn("last_name", trim(col("last_name"))) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("city", trim(col("city"))) \
    .withColumn("signup_date", trim(col("signup_date"))) \
    .withColumn("updated_at", to_timestamp(col("updated_at"))) \
    .filter(col("customer_id").isNotNull() & (col("customer_id") != "NULL") & (col("customer_id") != "")) \
    .filter(col("updated_at").isNotNull())

# Batch Level Deduplication (Latest updated_at standard pick karega)
window_spec = Window.partitionBy("customer_id").orderBy(col("updated_at").desc())

dedup_cleaned_df = cleaned_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

# Temp View for MERGE
dedup_cleaned_df.createOrReplaceTempView("dedup_cleaned_df_customers")

In [0]:
%sql
select * from dedup_cleaned_df_customers;

In [0]:
%sql
--  Step 1 ----------------------
merge into  cyntexa_dev.day_8.silver_customers_scd2  t
using dedup_cleaned_df_customers source
on t.customer_id = source.customer_id  and t.end_date is null
when matched and (
    t.email <> source.email or 
    t.city <> source.city
     )  then update 
     set 
t.end_date = source.updated_at,
t.is_current = false

WHEN NOT MATCHED THEN
  INSERT (
    customer_id, 
    first_name, 
    last_name, 
    email, 
    city, 
    signup_date, 
    start_date, 
    end_date, 
    is_current
  )
  VALUES (
    source.customer_id, 
    source.first_name, 
    source.last_name, 
    source.email, 
    source.city, 
    source.signup_date, 
    source.updated_at, 
    NULL, 
    true
  );








In [0]:
%sql
-- step 2---------------
INSERT INTO cyntexa_dev.day_8.silver_customers_scd2  (
  customer_id, first_name, last_name, email, city, signup_date, start_date, end_date, is_current
)
select s.customer_id , s.first_name, s.last_name, s.email, s.city, s.signup_date , s.updated_at, null, true from dedup_cleaned_df_customers  s   
left join  cyntexa_dev.day_8.silver_customers_scd2 t
on s.customer_id = t.customer_id and t.end_date is null  
where t.customer_id is null   


In [0]:
%sql 
select * from  cyntexa_dev.day_8.silver_customers_scd2 order by  customer_id;

# Question 5

Query the SCD Type 2 table to answer a point-in-time question, e.g. 'what was this customer's
address as of March 1st?'

In [0]:
%sql
SELECT *
FROM cyntexa_dev.day_8.silver_customers_scd2
WHERE customer_id = 'CUST101'
  AND TO_DATE(start_date) <= '2026-03-01'
  AND (TO_DATE(end_date) > '2026-03-01' OR end_date IS NULL);

#  Question 6

Compare the estimated DBU cost of running a job on all-purpose vs. job compute, and recommend
which Cyntexa should use for its nightly pipeline.

Cyntexa should strictly use **Job Compute (Automated Job Clusters)** for its nightly pipeline.

Switching from All-Purpose to Job Compute delivers an immediate **60% to 70% cost reduction** in Databricks Unit (DBU) consumption for identical workloads and hardware.

---

### DBU Cost Comparison

| Feature / Metric | All-Purpose Compute | Job Compute |
| --- | --- | --- |
| **Primary Use Case** | Interactive development, ad-hoc queries, notebook debugging | Scheduled production ETL, automated batch pipelines |
| **Approx. DBU Rate (Standard Tier)** | **~$0.40–$0.55 per DBU** | **~$0.15 per DBU** |
| **Cost Multiplier** | **~2.5x to 3.5x higher** | **Baseline (Lowest DBU rate)** |
| **Cluster Lifecycle** | Continuous / Long-running (Incurs idle costs) | Ephemeral (Spins up on job trigger, terminates immediately on completion) |

---

### Why Job Compute is the Right Choice for Cyntexa

* **Significant Cost Savings:** Databricks charges a premium for interactive features (like active notebook UI attachment and live user sessions). Automated nightly runs do not require these features.
* **Zero Idle Time Billing:** Job clusters automatically terminate the second the final pipeline task finishes, ensuring you never pay for unutilized cluster time.
* **Production Isolation:** Every nightly run starts with a clean, dedicated environment, eliminating memory leaks or resource contention caused by other developer activity on shared clusters.

---
